# 03. 학교 · 유치원 피처 엔지니어링 — 서울 아파트 최근 5년

[01_base_eda_feature_engineering.ipynb](01_base_eda_feature_engineering.ipynb)에서 만든
`df_fe_base.parquet`(전처리 + 기본/시간 피처 + 시장 lag 피처 + 좌표)을 불러와
유치원 · 초등학교 · 중학교 · 고등학교 최근접 거리 피처를 추가합니다.

지하철 피처는 별도로 [02_subway_feature_engineering.ipynb](02_subway_feature_engineering.ipynb)에서
같은 base 결과를 불러와 만들며, 두 결과의 병합은 이후 단계에서 진행합니다.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


# ------------------------------------------------------------------
# 경로 설정
# ------------------------------------------------------------------

# 현재 위치:
# Apartment price/notebook/v2_5year_notebook/eda
PROJECT_DIR = Path.cwd().parents[2]

DATA_ROOT = PROJECT_DIR / "data"

# V2 서울 5년 데이터
DATA_DIR = DATA_ROOT / "v2_5year_seoul"

PROCESSED_DIR = DATA_DIR / "processed"

# V1 / V2 공통 외부 데이터
EXTERNAL_DIR = DATA_ROOT / "external"


# ------------------------------------------------------------------
# 01_base_eda_feature_engineering.ipynb 결과 로드
# ------------------------------------------------------------------

DF_FE_BASE_PATH = PROCESSED_DIR / "df_fe_base.parquet"

if not DF_FE_BASE_PATH.exists():
    raise FileNotFoundError(
        f"기본 피처 데이터가 없습니다.\n"
        f"경로: {DF_FE_BASE_PATH}\n"
        f"먼저 01_base_eda_feature_engineering.ipynb를 실행하세요."
    )

df_fe = pd.read_parquet(DF_FE_BASE_PATH)

print("[입력 데이터]")
print("DF_FE_BASE_PATH :", DF_FE_BASE_PATH.resolve())
print("df_fe shape     :", df_fe.shape)


[입력 데이터]
DF_FE_BASE_PATH : C:\Users\UK\Apartment price\data\v2_5year_seoul\processed\df_fe_base.parquet
df_fe shape     : (234302, 69)


In [2]:
# ============================================================
# 기본 데이터 및 transaction_id 확인
# ============================================================

print("전체 거래 수:", len(df_fe))
print(
    "transaction_id 존재:",
    "transaction_id" in df_fe.columns
)
print(
    "transaction_id 고유 개수:",
    df_fe["transaction_id"].nunique()
)
print(
    "transaction_id 중복:",
    df_fe["transaction_id"].duplicated().sum()
)

전체 거래 수: 234302
transaction_id 존재: True
transaction_id 고유 개수: 234302
transaction_id 중복: 0


In [3]:
# ============================================================
# 지오코딩 공용 함수 (01_base_eda_feature_engineering.ipynb에서 이식)
# 유치원 좌표 fallback 검색에 사용
# ============================================================

import time

import requests
from dotenv import load_dotenv

load_dotenv()

KAKAO_API_KEY = os.environ.get(
    "KAKAO_REST_API_KEY",
    ""
)

if not KAKAO_API_KEY:
    raise ValueError(
        "KAKAO_REST_API_KEY 환경변수가 설정되지 않았습니다."
    )


def geocode_address(query: str):
    url = (
        "https://dapi.kakao.com/"
        "v2/local/search/address.json"
    )

    headers = {
        "Authorization": f"KakaoAK {KAKAO_API_KEY}"
    }

    params = {
        "query": query
    }

    try:
        resp = requests.get(
            url,
            headers=headers,
            params=params,
            timeout=5
        )

        resp.raise_for_status()

        docs = resp.json().get(
            "documents",
            []
        )

        if not docs:
            print(
                f"⚠ 검색 결과 없음: {query}"
            )
            return None, None

        lat = float(docs[0]["y"])
        lon = float(docs[0]["x"])

        return lat, lon

    except requests.RequestException as e:
        print(
            f"❌ API 요청 실패: {query}"
        )
        print(e)

        return None, None


GEOCODE_COLUMNS = ["검색주소", "위도", "경도"]


def load_geocode_cache(path):
    """저장된 지오코딩 결과를 불러온다."""
    if not path.exists():
        return pd.DataFrame(columns=GEOCODE_COLUMNS)

    cache = pd.read_csv(
        path,
        encoding="utf-8-sig"
    )

    return (
        cache[GEOCODE_COLUMNS]
        .drop_duplicates("검색주소", keep="last")
        .reset_index(drop=True)
    )


def collect_geocoding(
    addresses,
    cache_df,
    save_path,
    save_every=50,
    sleep_time=0.05
):
    results = []

    for i, address in enumerate(addresses, 1):
        lat, lon = geocode_address(address)

        results.append({
            "검색주소": address,
            "위도": lat,
            "경도": lon,
        })

        time.sleep(sleep_time)

        if i % save_every == 0 or i == len(addresses):
            new_data = pd.DataFrame(results)

            cache_df = (
                pd.concat(
                    [cache_df, new_data],
                    ignore_index=True
                )
                .drop_duplicates(
                    "검색주소",
                    keep="last"
                )
                .reset_index(drop=True)
            )

            cache_df.to_csv(
                save_path,
                index=False,
                encoding="utf-8-sig"
            )

            results = []

            print(
                f"진행: {i:,}/{len(addresses):,}"
            )

    return cache_df


## 8. 최근접 교육 인프라 거리

아파트와 교육시설의 위도·경도를 기반으로
초등학교, 중학교, 고등학교, 유치원의 최근접 거리를 계산합니다.

각 교육시설 유형별 최근접 거리를 미터(m) 단위로 생성하며,
주거 입지의 교육 인프라 접근성을 나타내는 피처로 활용합니다.

생성 피처:
- `초등학교거리_m`
- `중학교거리_m`
- `고등학교거리_m`
- `유치원거리_m`

### 5년 데이터에서의 시간적 주의사항

학교 및 유치원 위치 데이터가 현재 시점의 시설 목록 기준이라면,
과거 5년 동안의 신설·폐교·개원·폐원 이력을 완전히 반영하지 못할 수 있습니다.

따라서 현재 버전의 교육 인프라 거리 피처는
**현재 기준 생활·교육 인프라 접근성 proxy**로 사용합니다.

향후 설립일자·개원일자 및 폐교·폐원 이력 데이터가 확보되면,
거래 시점에 실제 운영 중이었던 시설만 후보에 포함하는
시점별 거리 계산으로 고도화할 수 있습니다.

8. 교육 인프라

8-1. 학교 데이터  
     ├─ 초등학교  
     ├─ 중학교  
     └─ 고등학교  

8-2. 유치원 데이터  

8-3. 최근접 교육시설 거리 생성  
     ├─ 초등학교거리_m  
     ├─ 중학교거리_m  
     ├─ 고등학교거리_m  
     └─ 유치원거리_m  

In [4]:
# ============================================================
# 학교 데이터 확인
# ============================================================

school_df = pd.read_csv(
    EXTERNAL_DIR / "school_locations.csv",
    encoding="utf-8-sig"
)

print(f"학교 데이터 shape : {school_df.shape}")

print("\n컬럼")
print(school_df.columns.tolist())

print("\n학교급별 개수")
display(
    school_df["학교급구분"]
    .value_counts()
    .to_frame("학교수")
)

학교 데이터 shape : (12011, 18)

컬럼
['학교ID', '학교명', '학교급구분', '설립일자', '설립형태', '본교분교구분', '운영상태', '소재지지번주소', '소재지도로명주소', '시도교육청코드', '시도교육청명', '교육지원청코드', '교육지원청명', '생성일자', '변경일자', '위도', '경도', '데이터기준일자']

학교급별 개수


,학교수
학교급구분,
초등학교,6303
중학교,3312
고등학교,2396


전국 학교 12,011개 중 초등학교 6,303 / 중학교 3,312 / 고등학교 2,396개가 포함되어 있어, 세 학교급을 모두 최근접 거리 피처로 만들 수 있는 데이터가 갖춰져 있습니다.

In [5]:
school_df["운영상태"].value_counts()

운영상태
운영    12011
Name: count, dtype: int64

전체 12,011개교가 모두 `운영` 상태입니다. 폐교 데이터가 이미 제외되어 있는 것으로 보여, 다음 셀에서 `운영상태 == "운영"` 필터는 방어적 코드로만 남기고 실질적으로는 걸러지는 행이 없습니다.

In [6]:
# ============================================================
# 서울 학교 데이터 정리
# ============================================================

seoul_school_df = (
    school_df.loc[
        school_df["시도교육청명"] == "서울특별시교육청"
    ]
    .copy()
)

for col in ["위도", "경도"]:
    seoul_school_df[col] = pd.to_numeric(
        seoul_school_df[col],
        errors="coerce"
    )

seoul_school_df = (
    seoul_school_df
    .dropna(subset=["위도", "경도"])
    .reset_index(drop=True)
)

print(f"서울 학교 수       : {len(seoul_school_df):,}")
print(
    "학교급별 개수"
)

display(
    seoul_school_df["학교급구분"]
    .value_counts()
    .to_frame("학교수")
)

서울 학교 수       : 1,313
학교급별 개수


,학교수
학교급구분,
초등학교,606
중학교,388
고등학교,319


In [7]:
# ============================================================
# 유치원 데이터 확인
# ============================================================

kindergarten_df = pd.read_csv(
    EXTERNAL_DIR / "kindergarten_locations.csv",
    encoding="cp949"
)

print(
    f"유치원 데이터 shape : "
    f"{kindergarten_df.shape}"
)

print("\n컬럼")
print(
    kindergarten_df.columns.tolist()
)

display(
    kindergarten_df.head()
)

유치원 데이터 shape : (946, 24)

컬럼
['교육청명', '교육지원청명', '유치원코드', '유치원명', '설립유형', '대표자명', '원장명', '설립일', '개원일', '주소', '전화번호', '홈페이지', '운영시간', '만3세학급수', '만4세학급수', '만5세학급수', '혼합학급수', '특수학급수', '만3세유아수', '만4세유아수', '만5세유아수', '혼합유아수', '특수유아수', '공시차수']


,교육청명,교육지원청명,유치원코드,유치원명,설립유형,대표자명,원장명,설립일,개원일,주소,...,만4세학급수,만5세학급수,혼합학급수,특수학급수,만3세유아수,만4세유아수,만5세유아수,혼합유아수,특수유아수,공시차수
0,서울특별시교육청,남부교육지원청,006b5238-ce41-4471-9ee3-4df8d24b5141,서울정심초등학교병설유치원,공립(병설),지선영,지선영,20181116,20190314,서울특별시 금천구 독산로78다길 89,...,1.0,1.0,0,1.0,10.0,11.0,18.0,0,3.0,20261
1,서울특별시교육청,강동송파교육지원청,009c017d-43bd-4d9f-ac64-d217abdd570e,서울솔방울유치원,공립(단설),장선화,장선화,20210301,20210301,서울특별시 송파구 오금로24길 25,...,2.0,2.0,0,1.0,28.0,36.0,40.0,0,0.0,20261
2,서울특별시교육청,강동송파교육지원청,04a021da-cdd7-4e7e-a434-2b388c2f6b63,란키즈유치원,사립(사인),송미화,송미화,20170307,20170307,서울특별시 송파구 새말로8길 22-5,...,1.0,1.0,0,NaN,46.0,24.0,27.0,0,NaN,20261
3,서울특별시교육청,동부교육지원청,0764f8c8-364c-495b-9fbc-b389148b7b4f,서울신현초등학교병설유치원,공립(병설),하옥선,하옥선,20201231,20210301,서울특별시 중랑구 봉화산로 188,...,1.0,1.0,0,1.0,11.0,13.0,14.0,0,2.0,20261
4,서울특별시교육청,동부교육지원청,076db651-036b-4bee-80ba-09f0edb4a57a,서울신묵초등학교병설유치원,공립(병설),김정윤,김정윤,20220301,20220301,서울특별시 중랑구 동일로149길 46,...,1.0,1.0,0,1.0,3.0,5.0,5.0,0,3.0,20261


In [8]:
# ============================================================
# 유치원 데이터 정리
# ============================================================

print("교육청")
display(
    kindergarten_df["교육청명"]
    .value_counts()
    .to_frame("유치원수")
)

print(
    "유치원 코드 수 :",
    kindergarten_df["유치원코드"].nunique()
)

print(
    "주소 결측      :",
    kindergarten_df["주소"].isna().sum()
)

교육청


,유치원수
교육청명,
서울특별시교육청,946


유치원 코드 수 : 946
주소 결측      : 0


In [9]:
# ============================================================
# 유치원 지오코딩 주소 생성
# ============================================================

kindergarten_df["검색주소"] = (
    kindergarten_df["주소"]
    .astype("string")
    .str.strip()
)

unique_kindergarten_addresses = (
    kindergarten_df[["검색주소"]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"고유 유치원 주소 : "
    f"{len(unique_kindergarten_addresses):,}"
)

고유 유치원 주소 : 924


In [10]:
# ============================================================
# 유치원 지오코딩 캐시 준비
# ============================================================

KINDERGARTEN_CACHE_PATH = (
    EXTERNAL_DIR / "kindergarten_geocode_cache.csv"
)

kindergarten_geo_cache = load_geocode_cache(
    KINDERGARTEN_CACHE_PATH
)

completed_addresses = set(
    kindergarten_geo_cache
    .dropna(subset=["위도", "경도"])
    ["검색주소"]
)

kindergarten_todo = (
    unique_kindergarten_addresses.loc[
        ~unique_kindergarten_addresses["검색주소"]
        .isin(completed_addresses),
        "검색주소"
    ]
    .tolist()
)

print(f"전체 주소 : {len(unique_kindergarten_addresses):,}")
print(f"처리 완료 : {len(completed_addresses):,}")
print(f"처리 예정 : {len(kindergarten_todo):,}")

전체 주소 : 924
처리 완료 : 915
처리 예정 : 9


In [11]:
# ============================================================
# 유치원 지오코딩 실행
# ============================================================

kindergarten_geo_cache = collect_geocoding(
    addresses=kindergarten_todo,
    cache_df=kindergarten_geo_cache,
    save_path=KINDERGARTEN_CACHE_PATH,
    save_every=50
)

⚠ 검색 결과 없음: 서울특별시 구로구 고척로25길 62
⚠ 검색 결과 없음: 서울특별시 강남구 선릉로10길 18-6
⚠ 검색 결과 없음: 서울특별시 양천구 오목로13길 31
⚠ 검색 결과 없음: 서울특별시 강서구 우현로 8
⚠ 검색 결과 없음: 서울특별시 금천구 탑골로3길 50
⚠ 검색 결과 없음: 서울특별시 관악구 솔밭로 20
⚠ 검색 결과 없음: 서울특별시 서대문구 통일로32길 23
⚠ 검색 결과 없음: 서울특별시 강동구 명일로 251
⚠ 검색 결과 없음: 서울특별시 관악구 조원로 114
진행: 9/9


In [12]:
# ============================================================
# 유치원명 기반 fallback 검색
# ============================================================

def search_kindergarten_by_name(query):
    """Kakao 키워드 검색을 수행한다."""

    url = "https://dapi.kakao.com/v2/local/search/keyword.json"

    headers = {
        "Authorization": f"KakaoAK {KAKAO_API_KEY}"
    }

    params = {
        "query": query,
        "size": 15,
    }

    try:
        response = requests.get(
            url,
            headers=headers,
            params=params,
            timeout=5,
        )
        response.raise_for_status()

        return response.json().get(
            "documents",
            []
        )

    except requests.RequestException:
        return []


def find_kindergarten_fallback(row):
    """여러 검색어를 시도하고 자치구가 일치하는 유치원을 선택한다."""

    name = row["유치원명"]
    address = row["주소"]
    district = address.split()[1]

    search_queries = [
        f"{name} {district}",
        name,
        f"{name} {address}",
        address,
    ]

    for query in search_queries:
        results = search_kindergarten_by_name(
            query
        )

        for result in results:
            result_address = (
                result.get("road_address_name")
                or result.get("address_name")
                or ""
            )

            if district not in result_address:
                continue

            return pd.Series({
                "검색주소": row["검색주소"],
                "유치원명": name,
                "검색어": query,
                "검색결과명": result.get("place_name"),
                "검색결과주소": result_address,
                "위도": float(result["y"]),
                "경도": float(result["x"]),
            })

    return pd.Series({
        "검색주소": row["검색주소"],
        "유치원명": name,
        "검색어": pd.NA,
        "검색결과명": pd.NA,
        "검색결과주소": pd.NA,
        "위도": pd.NA,
        "경도": pd.NA,
    })

In [13]:
# ============================================================
# 유치원 지오코딩 결과 확인
# ============================================================

success_mask = (
    kindergarten_geo_cache["위도"].notna()
    & kindergarten_geo_cache["경도"].notna()
)

total_count = len(kindergarten_geo_cache)
success_count = success_mask.sum()
fail_count = total_count - success_count

print(f"전체 주소 : {total_count:,}")
print(f"성공      : {success_count:,}")
print(f"실패      : {fail_count:,}")
print(f"성공률    : {success_count / total_count:.1%}")

전체 주소 : 924
성공      : 915
실패      : 9
성공률    : 99.0%


In [14]:
# ============================================================
# 유치원 지오코딩 실패 주소
# ============================================================

failed_kindergarten_geo = (
    kindergarten_geo_cache.loc[
        ~success_mask,
        ["검색주소"]
    ]
    .reset_index(drop=True)
)

print(
    f"실패 주소 : "
    f"{len(failed_kindergarten_geo):,}"
)

display(
    failed_kindergarten_geo
)

실패 주소 : 9


,검색주소
0,서울특별시 구로구 고척로25길 62
1,서울특별시 강남구 선릉로10길 18-6
2,서울특별시 양천구 오목로13길 31
3,서울특별시 강서구 우현로 8
4,서울특별시 금천구 탑골로3길 50
5,서울특별시 관악구 솔밭로 20
6,서울특별시 서대문구 통일로32길 23
7,서울특별시 강동구 명일로 251
8,서울특별시 관악구 조원로 114


In [15]:
# ============================================================
# 실패 유치원 원본 확인
# ============================================================

failed_kindergarten_detail = (
    kindergarten_df.loc[
        kindergarten_df["검색주소"].isin(
            failed_kindergarten_geo["검색주소"]
        ),
        [
            "유치원명",
            "설립유형",
            "주소",
            "검색주소",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

display(
    failed_kindergarten_detail
)

,유치원명,설립유형,주소,검색주소
0,서울아람유치원,사립(사인),서울특별시 구로구 고척로25길 62,서울특별시 구로구 고척로25길 62
1,신양유치원,사립(사인),서울특별시 강남구 선릉로10길 18-6,서울특별시 강남구 선릉로10길 18-6
2,대한유치원,사립(사인),서울특별시 양천구 오목로13길 31,서울특별시 양천구 오목로13길 31
3,문성유치원,사립(사인),서울특별시 강서구 우현로 8,서울특별시 강서구 우현로 8
4,백상유치원,사립(사인),서울특별시 금천구 탑골로3길 50,서울특별시 금천구 탑골로3길 50
5,종인유치원,사립(사인),서울특별시 관악구 솔밭로 20,서울특별시 관악구 솔밭로 20
6,인왕유치원,사립(사인),서울특별시 서대문구 통일로32길 23,서울특별시 서대문구 통일로32길 23
7,한성유치원,사립(사인),서울특별시 강동구 명일로 251,서울특별시 강동구 명일로 251
8,건영유치원,사립(법인),서울특별시 관악구 조원로 114,서울특별시 관악구 조원로 114


In [16]:
# ============================================================
# 실패 유치원 2차 검색어 생성
# ============================================================

def make_kindergarten_query(row):
    name = row["유치원명"]

    if "병설유치원" in name:
        name = name.replace(
            "병설유치원",
            ""
        )

    district = row["주소"].split()[1]

    return f"{name} {district}"


failed_kindergarten_detail["대체검색어"] = (
    failed_kindergarten_detail.apply(
        make_kindergarten_query,
        axis=1
    )
)

display(
    failed_kindergarten_detail[
        [
            "유치원명",
            "주소",
            "대체검색어",
        ]
    ]
)

,유치원명,주소,대체검색어
0,서울아람유치원,서울특별시 구로구 고척로25길 62,서울아람유치원 구로구
1,신양유치원,서울특별시 강남구 선릉로10길 18-6,신양유치원 강남구
2,대한유치원,서울특별시 양천구 오목로13길 31,대한유치원 양천구
3,문성유치원,서울특별시 강서구 우현로 8,문성유치원 강서구
4,백상유치원,서울특별시 금천구 탑골로3길 50,백상유치원 금천구
5,종인유치원,서울특별시 관악구 솔밭로 20,종인유치원 관악구
6,인왕유치원,서울특별시 서대문구 통일로32길 23,인왕유치원 서대문구
7,한성유치원,서울특별시 강동구 명일로 251,한성유치원 강동구
8,건영유치원,서울특별시 관악구 조원로 114,건영유치원 관악구


In [17]:
# ============================================================
# 실패 유치원 fallback 검색
# ============================================================

kindergarten_fallback = (
    failed_kindergarten_detail
    .apply(
        find_kindergarten_fallback,
        axis=1,
    )
)

display(kindergarten_fallback)

,검색주소,유치원명,검색어,검색결과명,검색결과주소,위도,경도
0,서울특별시 구로구 고척로25길 62,서울아람유치원,<NA>,<NA>,<NA>,<NA>,<NA>
1,서울특별시 강남구 선릉로10길 18-6,신양유치원,<NA>,<NA>,<NA>,<NA>,<NA>
2,서울특별시 양천구 오목로13길 31,대한유치원,<NA>,<NA>,<NA>,<NA>,<NA>
3,서울특별시 강서구 우현로 8,문성유치원,<NA>,<NA>,<NA>,<NA>,<NA>
4,서울특별시 금천구 탑골로3길 50,백상유치원,<NA>,<NA>,<NA>,<NA>,<NA>
5,서울특별시 관악구 솔밭로 20,종인유치원,<NA>,<NA>,<NA>,<NA>,<NA>
6,서울특별시 서대문구 통일로32길 23,인왕유치원,<NA>,<NA>,<NA>,<NA>,<NA>
7,서울특별시 강동구 명일로 251,한성유치원,<NA>,<NA>,<NA>,<NA>,<NA>
8,서울특별시 관악구 조원로 114,건영유치원,<NA>,<NA>,<NA>,<NA>,<NA>


In [18]:
# ============================================================
# 2차 검색 성공 결과 반영
# ============================================================

fallback_success = (
    kindergarten_fallback
    .dropna(subset=["위도", "경도"])
    [["검색주소", "위도", "경도"]]
    .drop_duplicates(
        "검색주소",
        keep="last"
    )
)

kindergarten_geo_cache = (
    pd.concat(
        [
            kindergarten_geo_cache,
            fallback_success,
        ],
        ignore_index=True
    )
    .drop_duplicates(
        "검색주소",
        keep="last"
    )
    .reset_index(drop=True)
)

kindergarten_geo_cache.to_csv(
    KINDERGARTEN_CACHE_PATH,
    index=False,
    encoding="utf-8-sig"
)

In [19]:
# ============================================================
# 남은 실패 주소 확인
# ============================================================

success_mask = (
    kindergarten_geo_cache["위도"].notna()
    & kindergarten_geo_cache["경도"].notna()
)

remaining_failed = (
    kindergarten_geo_cache.loc[
        ~success_mask,
        ["검색주소"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"남은 실패 주소 : "
    f"{len(remaining_failed):,}"
)

display(remaining_failed)

남은 실패 주소 : 9


,검색주소
0,서울특별시 구로구 고척로25길 62
1,서울특별시 강남구 선릉로10길 18-6
2,서울특별시 양천구 오목로13길 31
3,서울특별시 강서구 우현로 8
4,서울특별시 금천구 탑골로3길 50
5,서울특별시 관악구 솔밭로 20
6,서울특별시 서대문구 통일로32길 23
7,서울특별시 강동구 명일로 251
8,서울특별시 관악구 조원로 114


In [20]:
# ============================================================
# 남은 지오코딩 실패 유치원 확인
# ============================================================

remaining_detail = (
    kindergarten_df.loc[
        kindergarten_df["검색주소"].isin(
            remaining_failed["검색주소"]
        ),
        [
            "유치원명",
            "설립유형",
            "주소",
            "검색주소",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

display(remaining_detail)

,유치원명,설립유형,주소,검색주소
0,서울아람유치원,사립(사인),서울특별시 구로구 고척로25길 62,서울특별시 구로구 고척로25길 62
1,신양유치원,사립(사인),서울특별시 강남구 선릉로10길 18-6,서울특별시 강남구 선릉로10길 18-6
2,대한유치원,사립(사인),서울특별시 양천구 오목로13길 31,서울특별시 양천구 오목로13길 31
3,문성유치원,사립(사인),서울특별시 강서구 우현로 8,서울특별시 강서구 우현로 8
4,백상유치원,사립(사인),서울특별시 금천구 탑골로3길 50,서울특별시 금천구 탑골로3길 50
5,종인유치원,사립(사인),서울특별시 관악구 솔밭로 20,서울특별시 관악구 솔밭로 20
6,인왕유치원,사립(사인),서울특별시 서대문구 통일로32길 23,서울특별시 서대문구 통일로32길 23
7,한성유치원,사립(사인),서울특별시 강동구 명일로 251,서울특별시 강동구 명일로 251
8,건영유치원,사립(법인),서울특별시 관악구 조원로 114,서울특별시 관악구 조원로 114


In [21]:
# ============================================================
# 자치구 일치 결과만 선택
# ============================================================

def find_kindergarten_fallback(row):
    district = row["주소"].split()[1]

    results = search_kindergarten_by_name(
        row["유치원명"]
    )

    for result in results:
        address = (
            result.get("road_address_name")
            or result.get("address_name")
            or ""
        )

        if district in address:
            return pd.Series({
                "검색주소": row["검색주소"],
                "유치원명": row["유치원명"],
                "검색결과명": result.get("place_name"),
                "검색결과주소": address,
                "위도": float(result["y"]),
                "경도": float(result["x"]),
            })

    return pd.Series({
        "검색주소": row["검색주소"],
        "유치원명": row["유치원명"],
        "검색결과명": pd.NA,
        "검색결과주소": pd.NA,
        "위도": pd.NA,
        "경도": pd.NA,
    })

In [22]:
# ============================================================
# 남은 9개 유치원 지번주소 fallback
# ============================================================

manual_queries = {
    "서울특별시 구로구 고척로25길 62": [
        "서울특별시 구로구 개봉동 115-10",
        "서울아람유치원 개봉동",
    ],
    "서울특별시 강남구 선릉로10길 18-6": [
        "서울특별시 강남구 개포동 164-2",
        "신양유치원 개포동",
    ],
    "서울특별시 양천구 오목로13길 31": [
        "대한유치원 신월동",
        "서울특별시 양천구 신월동 대한유치원",
    ],
    "서울특별시 강서구 우현로 8": [
        "서울특별시 강서구 화곡동 94-1",
        "문성유치원 화곡동",
    ],
    "서울특별시 금천구 탑골로3길 50": [
        "서울특별시 금천구 시흥동 220-2",
        "백상유치원 시흥동",
    ],
    "서울특별시 관악구 솔밭로 20": [
        "서울특별시 관악구 봉천동 1690-17",
        "종인유치원 봉천동",
    ],
    "서울특별시 서대문구 통일로32길 23": [
        "서울특별시 서대문구 홍제동 105-88",
        "인왕유치원 홍제동",
    ],
    "서울특별시 강동구 명일로 251": [
        "서울특별시 강동구 길동 91-12",
        "한성유치원 길동",
    ],
    "서울특별시 관악구 조원로 114": [
        "서울특별시 관악구 신림동 513-17",
        "건영유치원 신림동",
    ],
}


manual_results = []

for address, queries in manual_queries.items():
    lat = None
    lon = None
    matched_query = None

    for query in queries:
        # 1. 주소 검색
        lat, lon = geocode_address(query)

        if lat is not None:
            matched_query = query
            break

        # 2. 키워드 검색
        results = search_kindergarten_by_name(query)

        if results:
            lat = float(results[0]["y"])
            lon = float(results[0]["x"])
            matched_query = query
            break

    manual_results.append({
        "검색주소": address,
        "성공검색어": matched_query,
        "위도": lat,
        "경도": lon,
    })


manual_geo = pd.DataFrame(manual_results)

display(manual_geo)

print(
    "남은 실패:",
    manual_geo[["위도", "경도"]]
    .isna()
    .any(axis=1)
    .sum()
)

⚠ 검색 결과 없음: 대한유치원 신월동
⚠ 검색 결과 없음: 서울특별시 양천구 신월동 대한유치원
⚠ 검색 결과 없음: 서울특별시 관악구 봉천동 1690-17
⚠ 검색 결과 없음: 종인유치원 봉천동


,검색주소,성공검색어,위도,경도
0,서울특별시 구로구 고척로25길 62,서울특별시 구로구 개봉동 115-10,37.504486,126.846238
1,서울특별시 강남구 선릉로10길 18-6,서울특별시 강남구 개포동 164-2,37.483548,127.061600
2,서울특별시 양천구 오목로13길 31,None,NaN,NaN
3,서울특별시 강서구 우현로 8,서울특별시 강서구 화곡동 94-1,37.547083,126.844604
4,서울특별시 금천구 탑골로3길 50,서울특별시 금천구 시흥동 220-2,37.453583,126.913651
5,서울특별시 관악구 솔밭로 20,None,NaN,NaN
6,서울특별시 서대문구 통일로32길 23,서울특별시 서대문구 홍제동 105-88,37.585413,126.949500
7,서울특별시 강동구 명일로 251,서울특별시 강동구 길동 91-12,37.541176,127.146880
8,서울특별시 관악구 조원로 114,서울특별시 관악구 신림동 513-17,37.484792,126.914953


남은 실패: 2


In [23]:
# ============================================================
# 마지막 2개 유치원 지번주소 지오코딩
# ============================================================

last_manual_queries = {
    "서울특별시 양천구 오목로13길 31": [
        "서울특별시 양천구 신월동 465-6",
        "서울특별시 양천구 월정로8길 23",
    ],
    "서울특별시 관악구 솔밭로 20": [
        "서울특별시 관악구 봉천동 1690-17",
    ],
}

last_manual_results = []

for original_address, queries in last_manual_queries.items():
    lat = None
    lon = None
    success_query = None

    for query in queries:
        lat, lon = geocode_address(query)

        if lat is not None:
            success_query = query
            break

    last_manual_results.append({
        "검색주소": original_address,
        "성공검색어": success_query,
        "위도": lat,
        "경도": lon,
    })

last_manual_geo = pd.DataFrame(
    last_manual_results
)

display(last_manual_geo)

print(
    "남은 실패:",
    last_manual_geo[["위도", "경도"]]
    .isna()
    .any(axis=1)
    .sum()
)

⚠ 검색 결과 없음: 서울특별시 관악구 봉천동 1690-17


,검색주소,성공검색어,위도,경도
0,서울특별시 양천구 오목로13길 31,서울특별시 양천구 신월동 465-6,37.525719,126.844798
1,서울특별시 관악구 솔밭로 20,None,NaN,NaN


남은 실패: 1


In [24]:
kindergarten_manual_geo = pd.DataFrame({
    "검색주소": [
        # 검증된 주소
    ],
    "위도": [
        # 검증된 위도
    ],
    "경도": [
        # 검증된 경도
    ],
})

In [25]:
kindergarten_df = kindergarten_df.merge(
    kindergarten_geo_cache[
        [
            "검색주소",
            "위도",
            "경도",
        ]
    ],
    on="검색주소",
    how="left",
    validate="many_to_one"
)

print(
    "좌표 결측:",
    kindergarten_df[
        ["위도", "경도"]
    ]
    .isna()
    .any(axis=1)
    .sum()
)

좌표 결측: 9


In [26]:
# ============================================================
# 거리 계산용 유치원 데이터
# ============================================================

kindergarten_valid_df = (
    kindergarten_df
    .dropna(subset=["위도", "경도"])
    .reset_index(drop=True)
)

print(f"전체 유치원       : {len(kindergarten_df):,}")
print(f"좌표 확인 유치원  : {len(kindergarten_valid_df):,}")
print(
    f"제외 유치원       : "
    f"{len(kindergarten_df) - len(kindergarten_valid_df):,}"
)

전체 유치원       : 946
좌표 확인 유치원  : 937
제외 유치원       : 9


현재 지도 검색에서 위치를 확인할 수 없는 유치원 9개는
현재 시점 교육 인프라 접근성 계산 대상에서 제외하였다.

# 거리 피처 생성

In [27]:
# ============================================================
# 교육시설 데이터 통합
# ============================================================

school_facilities = (
    seoul_school_df[
        [
            "학교명",
            "학교급구분",
            "위도",
            "경도",
        ]
    ]
    .rename(
        columns={
            "학교명": "시설명",
            "학교급구분": "시설구분",
        }
    )
    .copy()
)

kindergarten_facilities = (
    kindergarten_df[
        [
            "유치원명",
            "위도",
            "경도",
        ]
    ]
    .rename(
        columns={
            "유치원명": "시설명",
        }
    )
    .copy()
)

kindergarten_facilities["시설구분"] = "유치원"

education_df = pd.concat(
    [
        kindergarten_facilities,
        school_facilities,
    ],
    ignore_index=True,
)

education_df = (
    education_df
    .dropna(subset=["위도", "경도"])
    .reset_index(drop=True)
)

display(
    education_df["시설구분"]
    .value_counts()
    .to_frame("시설수")
)

print(
    f"\n전체 교육시설 수 : "
    f"{len(education_df):,}"
)

,시설수
시설구분,
유치원,937
초등학교,606
중학교,388
고등학교,319



전체 교육시설 수 : 2,250


In [28]:
education_df

,시설명,위도,경도,시설구분
0,서울정심초등학교병설유치원,37.471983,126.908187,유치원
1,서울솔방울유치원,37.504849,127.117385,유치원
2,란키즈유치원,37.480798,127.126295,유치원
3,서울신현초등학교병설유치원,37.605815,127.093522,유치원
4,서울신묵초등학교병설유치원,37.606523,127.075836,유치원
...,...,...,...,...
2245,서울하늘숲초등학교,37.484181,126.841652,초등학교
2246,서울개현초등학교,37.480788,127.061627,초등학교
2247,서울고일초등학교,37.551543,127.169410,초등학교
2248,환일고등학교,37.553607,126.962802,고등학교


In [29]:
# ============================================================
# 최근접 시설 계산 함수
# ============================================================

from sklearn.neighbors import BallTree
import numpy as np

EARTH_RADIUS_M = 6_371_000


def nearest_facility(
    source_lat,
    source_lon,
    target_lat,
    target_lon,
    target_names,
):
    """각 위치에서 가장 가까운 시설명과 거리를 계산한다."""

    source_coords = np.radians(
        np.column_stack([
            source_lat,
            source_lon,
        ])
    )

    target_coords = np.radians(
        np.column_stack([
            target_lat,
            target_lon,
        ])
    )

    tree = BallTree(
        target_coords,
        metric="haversine",
    )

    distances, indices = tree.query(
        source_coords,
        k=1,
    )

    nearest_names = target_names[
        indices[:, 0]
    ]

    nearest_distances = (
        distances[:, 0]
        * EARTH_RADIUS_M
    )

    return nearest_names, nearest_distances

In [30]:
# ============================================================
# 교육시설 거리 피처 생성 함수
# ============================================================

def add_education_distance(
    df,
    education_df,
    facility_type,
):
    """시설 유형별 최근접 시설명과 거리 피처를 추가한다."""

    apt_mask = (
        df["위도"].notna()
        & df["경도"].notna()
    )

    target_df = (
        education_df.loc[
            education_df["시설구분"] == facility_type
        ]
        .dropna(subset=["위도", "경도"])
        .reset_index(drop=True)
    )

    names, distances = nearest_facility(
        df.loc[apt_mask, "위도"].to_numpy(),
        df.loc[apt_mask, "경도"].to_numpy(),
        target_df["위도"].to_numpy(),
        target_df["경도"].to_numpy(),
        target_df["시설명"].to_numpy(),
    )

    df.loc[
        apt_mask,
        f"최근접{facility_type}"
    ] = names

    df.loc[
        apt_mask,
        f"{facility_type}거리_m"
    ] = distances

In [31]:
# ============================================================
# 교육 인프라 거리 피처 생성
# ============================================================

education_types = [
    "유치원",
    "초등학교",
    "중학교",
    "고등학교",
]

for facility_type in education_types:
    add_education_distance(
        df_fe,
        education_df,
        facility_type,
    )

In [32]:
# ============================================================
# 교육 인프라 거리 피처 검증
# ============================================================

education_distance_cols = [
    "유치원거리_m",
    "초등학교거리_m",
    "중학교거리_m",
    "고등학교거리_m",
]

display(
    df_fe[
        education_distance_cols
    ]
    .describe()
    .T
)

print("\n결측 건수")

display(
    df_fe[
        education_distance_cols
    ]
    .isna()
    .sum()
    .to_frame("결측건수")
)

,count,mean,std,min,25%,50%,75%,max
유치원거리_m,234302.0,277.457905,165.344566,3.882308,157.776589,242.207325,368.293352,1330.590566
초등학교거리_m,234302.0,327.302267,155.782654,23.319600,207.831838,300.429989,420.021744,1768.785450
중학교거리_m,234302.0,465.116713,239.863885,24.066877,279.368480,428.630715,608.550402,2066.885015
고등학교거리_m,234302.0,607.920882,349.695174,27.363768,347.996717,537.361888,793.021992,2908.389959



결측 건수


,결측건수
유치원거리_m,0
초등학교거리_m,0
중학교거리_m,0
고등학교거리_m,0


In [34]:
education_cols = [
    col for col in df_fe.columns
    if (
        "학교" in col
        or "유치원" in col
    )
]

print("교육 피처:")
print(education_cols)

print("\n결측치:")
print(df_fe[education_cols].isna().sum())

display(
    df_fe[
        ["transaction_id"] + education_cols
    ].head()
)

교육 피처:
['최근접유치원', '유치원거리_m', '최근접초등학교', '초등학교거리_m', '최근접중학교', '중학교거리_m', '최근접고등학교', '고등학교거리_m']

결측치:
최근접유치원      0
유치원거리_m     0
최근접초등학교     0
초등학교거리_m    0
최근접중학교      0
중학교거리_m     0
최근접고등학교     0
고등학교거리_m    0
dtype: int64


,transaction_id,최근접유치원,유치원거리_m,최근접초등학교,초등학교거리_m,최근접중학교,중학교거리_m,최근접고등학교,고등학교거리_m
0,0,린덴바움유치원,105.912487,서울독립문초등학교,184.266445,대신중학교,280.863059,대신고등학교,279.276035
1,1,세종유치원,149.357863,서울매동초등학교,382.997401,배화여자중학교,587.017383,배화여자고등학교,587.017383
2,2,서울금화초등학교병설유치원,215.938715,서울금화초등학교,215.940049,동명여자중학교,260.398027,대신고등학교,340.454004
3,3,세종유치원,369.059093,서울덕수초등학교,444.822543,창덕여자중학교,738.474944,이화여자고등학교,720.719753
4,4,서울명신초등학교병설유치원,206.178265,서울명신초등학교,280.621824,한성여자중학교,304.879303,서일문화예술고등학교,135.613244


In [35]:
# ============================================================
# 학교 · 유치원 피처 데이터 저장
# ============================================================

DF_FE_SCHOOL_PATH = (
    PROCESSED_DIR / "df_fe_school.parquet"
)

school_feature_cols = [
    "transaction_id",
    "최근접유치원",
    "유치원거리_m",
    "최근접초등학교",
    "초등학교거리_m",
    "최근접중학교",
    "중학교거리_m",
    "최근접고등학교",
    "고등학교거리_m",
]

df_fe_school = (
    df_fe[school_feature_cols]
    .copy()
)

# 저장 전 검증
assert len(df_fe_school) == len(df_fe)
assert df_fe_school["transaction_id"].is_unique

df_fe_school.to_parquet(
    DF_FE_SCHOOL_PATH,
    index=False,
)

print("[학교 · 유치원 피처 저장 완료]")
print("저장 경로:", DF_FE_SCHOOL_PATH.resolve())
print("저장 shape:", df_fe_school.shape)

display(df_fe_school.head())

[학교 · 유치원 피처 저장 완료]
저장 경로: C:\Users\UK\Apartment price\data\v2_5year_seoul\processed\df_fe_school.parquet
저장 shape: (234302, 9)


,transaction_id,최근접유치원,유치원거리_m,최근접초등학교,초등학교거리_m,최근접중학교,중학교거리_m,최근접고등학교,고등학교거리_m
0,0,린덴바움유치원,105.912487,서울독립문초등학교,184.266445,대신중학교,280.863059,대신고등학교,279.276035
1,1,세종유치원,149.357863,서울매동초등학교,382.997401,배화여자중학교,587.017383,배화여자고등학교,587.017383
2,2,서울금화초등학교병설유치원,215.938715,서울금화초등학교,215.940049,동명여자중학교,260.398027,대신고등학교,340.454004
3,3,세종유치원,369.059093,서울덕수초등학교,444.822543,창덕여자중학교,738.474944,이화여자고등학교,720.719753
4,4,서울명신초등학교병설유치원,206.178265,서울명신초등학교,280.621824,한성여자중학교,304.879303,서일문화예술고등학교,135.613244
